# Marketing Mix Modeling Agents

Full-stack example showing a PyMC-based marketing mix model (MMM) with synthetic adstocked media data, business-aware evaluation,
and a reinforcement learning loop that tunes hyperparameters using a composite score blending statistical fit with marketing heuristics.

In [37]:
# Cell 1: imports, synthetic dataset, and shared resources

import math
import os
from dataclasses import dataclass
from typing import Any, Mapping, Sequence
from getpass import getpass

import numpy as np
import pandas as pd

SETUP_WARNINGS: list[str] = []

try:
    import pymc as pm
    from pytensor import tensor as pt
    PYMC_AVAILABLE = True
except Exception:
    pm = None
    pt = None
    PYMC_AVAILABLE = False
    SETUP_WARNINGS.append("PyMC unavailable; falling back to deterministic solver.")

from agentic_codex import AgentBuilder, Context, FunctionAdapter
from agentic_codex.core.capabilities import ContextCapability, MessageBusCapability, ResourceCapability
from agentic_codex.core.message_bus import MessageBus
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.llm import EnvOpenAIAdapter


WEEKS = 104
RNG = np.random.default_rng(86)
CHANNELS = [
    {"name": "search", "base": 120.0, "elasticity": 0.065, "share": 0.28},
    {"name": "social", "base": 90.0, "elasticity": 0.055, "share": 0.22},
    {"name": "tv", "base": 150.0, "elasticity": 0.045, "share": 0.32},
    {"name": "affiliate", "base": 60.0, "elasticity": 0.035, "share": 0.18},
]

@dataclass
class MediaConfig:
    adstock_rate: float
    hill_alpha: float
    hill_gamma: float

def apply_adstock(spend: np.ndarray, rate: float) -> np.ndarray:
    carry = 0.0
    transformed = np.zeros_like(spend, dtype=float)
    for idx, value in enumerate(spend):
        carry = value + rate * carry
        transformed[idx] = carry
    return transformed

def apply_hill_function(exposure: np.ndarray, alpha: float, gamma: float) -> np.ndarray:
    exposure_alpha = np.power(np.maximum(exposure, 0.0), alpha)
    gamma_alpha = gamma ** alpha
    return exposure_alpha / (exposure_alpha + gamma_alpha + 1e-8)

def generate_marketing_data(weeks: int, channels: Sequence[Mapping[str, Any]], config: MediaConfig) -> pd.DataFrame:
    trend = 500 + 2.5 * np.arange(weeks)
    seasonal = 40 * np.sin(np.linspace(0, 6 * math.pi, weeks))
    noise = RNG.normal(0, 30, size=weeks)
    raw_spend = {}
    transformed = {}
    for channel in channels:
        spend = RNG.gamma(shape=5, scale=channel["share"] * 80, size=weeks)
        spend += np.linspace(0, channel["share"] * 20, weeks)
        raw_spend[f"{channel['name']}_spend"] = spend
        adstocked = apply_adstock(spend, config.adstock_rate)
        transformed[channel["name"]] = apply_hill_function(adstocked, config.hill_alpha, config.hill_gamma)
    base_sales = trend + seasonal + noise
    contribution = np.zeros(weeks)
    for channel in channels:
        contribution += channel["elasticity"] * transformed[channel["name"]] * 1000
    sales = base_sales + contribution
    df = pd.DataFrame({"week": np.arange(weeks), "sales": sales})
    for key, value in raw_spend.items():
        df[key] = value
    for channel in channels:
        df[f"{channel['name']}_transformed"] = transformed[channel["name"]]
    return df

BASE_MEDIA_CONFIG = MediaConfig(adstock_rate=0.45, hill_alpha=1.25, hill_gamma=0.7)
MARKETING_DATAFRAME = generate_marketing_data(WEEKS, CHANNELS, BASE_MEDIA_CONFIG)
MARKETING_DATA = MARKETING_DATAFRAME.to_dict(orient="records")

MARKETING_HYPERPARAMS = {
    "media": {
        "adstock_rate": BASE_MEDIA_CONFIG.adstock_rate,
        "hill_alpha": BASE_MEDIA_CONFIG.hill_alpha,
        "hill_gamma": BASE_MEDIA_CONFIG.hill_gamma,
    },
    "sampler": {"draws": 1000, "tune": 500, "chains": 1},
    "priors": {
        "intercept": {"mean": 400.0, "sd": 100.0},
        "sigma": {"sd": 50.0},
        "channels": {channel["name"]: {"scale": channel["elasticity"] * 2.0, "positive": True} for channel in CHANNELS},
    },
}

EVALUATION_KB = {
    "fit_quality": [
        "Prefer RMSE < 40 and MAPE < 10% for weekly MMM fits.",
        "Posterior sigma above 80 usually signals underfit or omitted seasonality.",
    ],
    "sign_expectations": [
        "Search and social elasticities should stay positive; negative draws violate business priors.",
        "TV often drives higher incremental reach with diminishing returns.",
    ],
    "drivers": [
        "Elasticities between 0.03 and 0.08 align with blended funnels.",
        "ROI > 1.5 indicates scalable efficiency; < 1 suggests over-saturation.",
    ],
}

RECOMMENDATION_KB = {
    "budgeting": [
        "Shift incremental budget toward channels with ROI multiples and available headroom.",
        "Throttle channels with high saturation indices before reallocating spend.",
    ],
    "operations": [
        "Coordinate creative refreshes when TV carry-over drifts beyond 3 weeks.",
        "Raise bid caps for search when ROI gap to target exceeds 0.2.",
    ],
    "heuristics": [
        "Search saturates slower than social; emphasise always-on investments.",
        "Maintain at least 20% of budget in awareness channels to preserve baseline.",
    ],
}

def build_context_capability() -> ContextCapability:
    return ContextCapability(
        data={
            "dataset": MARKETING_DATA,
            "channels": [ch["name"] for ch in CHANNELS],
            "hyperparameters": MARKETING_HYPERPARAMS,
            "warnings": list(SETUP_WARNINGS),
        }
    )

def build_message_bus() -> tuple[MessageBus, MessageBusCapability]:
    bus = MessageBus()
    return bus, MessageBusCapability(bus=bus)

def build_evaluation_resource() -> ResourceCapability:
    return ResourceCapability(name="knowledge.evaluation", resource=EVALUATION_KB, target="components")

def build_recommendation_resource() -> ResourceCapability:
    return ResourceCapability(name="knowledge.recommendation", resource=RECOMMENDATION_KB, target="components")

MARKETING_RL_ENVIRONMENT = {
    "target_score": 0.78,
    "weights": {"stats": 0.6, "roi": 0.25, "sign": 0.15},
    "rmse_baseline": 80.0,
    "roi_target": 1.6,
    "adstock_bounds": (0.2, 0.9),
    "hill_alpha_bounds": (0.8, 2.2),
    "hill_gamma_bounds": (0.3, 1.2),
    "step_sizes": {"adstock": 0.04, "alpha": 0.1, "gamma": 0.05},
}

OPENAI_MODEL = "gpt-4o-mini"
if not os.getenv("OPENAI_API_KEY"):
    try:
        os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
    except (EOFError, KeyboardInterrupt):
        os.environ["OPENAI_API_KEY"] = ""

try:
    candidate_llm = EnvOpenAIAdapter(model=OPENAI_MODEL)
    if getattr(candidate_llm, "_client", None) is None:
        raise RuntimeError("OpenAI backend unavailable")
    llm_adapter = candidate_llm
except Exception:
    def _fallback(prompt: str) -> str:
        lines = [line.strip() for line in prompt.splitlines() if line.strip()]
        excerpt = lines[-1][:200] if lines else "MMM analysis pending completion"
        return f"[stub llm] {excerpt}"
    llm_adapter = FunctionAdapter(_fallback)
    SETUP_WARNINGS.append("OpenAI adapter unavailable; using stub LLM.")


In [39]:

# Cell 2: PyMC modelling helpers and agent step

from typing import Dict, List

def build_design_matrix(records: Sequence[Mapping[str, Any]], channels: Sequence[str], hyperparams: Mapping[str, Any]) -> tuple[np.ndarray, np.ndarray, Dict[str, np.ndarray]]:
    df = pd.DataFrame(records)
    adstock_rate = hyperparams["media"]["adstock_rate"]
    alpha = hyperparams["media"]["hill_alpha"]
    gamma = hyperparams["media"]["hill_gamma"]
    exposures: Dict[str, np.ndarray] = {}
    for channel in channels:
        spend = df[f"{channel}_spend"].to_numpy(dtype=float)
        adstocked = apply_adstock(spend, adstock_rate)
        exposures[channel] = apply_hill_function(adstocked, alpha, gamma)
    X = np.column_stack([exposures[channel] for channel in channels])
    y = df["sales"].to_numpy(dtype=float)
    return X, y, exposures


def record_warning(ctx: Context, message: str | None = None) -> List[str]:
    warnings = ctx.scratch.setdefault("warnings", list(ctx.scratch.get("context", {}).get("warnings", [])))
    if message and message not in warnings:
        warnings.append(message)
    return warnings


def ensure_sampler_settings(ctx: Context, hyperparams: Mapping[str, Any]) -> None:
    sampler = hyperparams.setdefault("sampler", {})
    chains = int(sampler.get("chains", 1))
    if PYMC_AVAILABLE and chains < 2:
        sampler["chains"] = 2
        record_warning(ctx, "Sampler chains increased to 2 to encourage convergence.")


def run_mmm_inference(
    X: np.ndarray,
    y: np.ndarray,
    channels: Sequence[str],
    priors: Mapping[str, Any],
    sampler_cfg: Mapping[str, Any],
) -> Mapping[str, Any]:
    if PYMC_AVAILABLE:
        with pm.Model() as model:
            intercept = pm.Normal("intercept", mu=priors["intercept"]["mean"], sigma=priors["intercept"]["sd"])
            sigma = pm.HalfNormal("sigma", sigma=priors["sigma"]["sd"])
            betas = []
            for channel in channels:
                prior_cfg = priors["channels"][channel]
                if prior_cfg.get("positive", False):
                    beta = pm.HalfNormal(channel, sigma=prior_cfg["scale"] + 1e-6)
                else:
                    beta = pm.Normal(channel, mu=0.0, sigma=prior_cfg["scale"] + 1e-6)
                betas.append(beta)
            mu = intercept + pm.math.dot(X, pt.stack(betas))
            pm.Normal("sales", mu=mu, sigma=sigma, observed=y)
            step = pm.Metropolis()
            trace = pm.sample(
                draws=sampler_cfg.get("draws", 1000),
                tune=sampler_cfg.get("tune", 500),
                chains=sampler_cfg.get("chains", 1),
                step=step,
                random_seed=RNG.integers(1, 10_000),
                progressbar=False,
                return_inferencedata=False,
            )
        posterior_intercept = float(trace["intercept"].mean())
        posterior_sigma = float(trace["sigma"].mean())
        coefficients = {channel: float(trace[channel].mean()) for channel in channels}
        predictions = posterior_intercept + X.dot(np.array(list(coefficients.values())))
        posterior = {"intercept": posterior_intercept, "sigma": posterior_sigma}
    else:
        XtX = X.T @ X + np.eye(X.shape[1]) * 1e-6
        Xty = X.T @ y
        coeff_vector = np.linalg.solve(XtX, Xty)
        intercept = float(np.mean(y) - np.mean(X, axis=0) @ coeff_vector)
        coefficients = {channel: max(0.0, coeff_vector[idx]) for idx, channel in enumerate(channels)}
        predictions = intercept + X @ np.array(list(coefficients.values()))
        posterior = {"intercept": intercept, "sigma": float(np.std(y - predictions))}
    predictions = np.asarray(predictions, dtype=float)
    residuals = y - predictions
    rmse = float(np.sqrt(np.mean(residuals ** 2)))
    mae = float(np.mean(np.abs(residuals)))
    mape = float(np.mean(np.abs(residuals / np.maximum(y, 1e-6))) * 100)
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    ss_res = float(np.sum(residuals ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot else 0.0
    return {
        "coefficients": coefficients,
        "posterior": posterior,
        "predictions": predictions.tolist(),
        "metrics": {"rmse": rmse, "mae": mae, "mape": mape, "r2": r2},
        "residuals": residuals.tolist(),
    }


def compute_business_metrics(df: pd.DataFrame, exposures: Mapping[str, np.ndarray], coefficients: Mapping[str, float]) -> Mapping[str, Any]:
    roi = {}
    sign_alignment = {}
    spend_totals = {}
    for channel, beta in coefficients.items():
        spend = df[f"{channel}_spend"].to_numpy(dtype=float)
        exposure = exposures[channel]
        incremental = beta * exposure * 1000
        roi[channel] = float(np.sum(incremental) / np.sum(np.maximum(spend, 1e-6)))
        spend_totals[channel] = float(np.sum(spend))
        sign_alignment[channel] = beta >= 0
    avg_roi = float(np.mean(list(roi.values()))) if roi else 0.0
    saturation_index = {channel: float(np.mean(exposures[channel])) for channel in exposures}
    return {
        "roi": roi,
        "avg_roi": avg_roi,
        "sign_alignment": sign_alignment,
        "saturation_index": saturation_index,
        "spend_totals": spend_totals,
    }


def modelling_step(ctx: Context) -> AgentStep:
    dataset = ctx.scratch["context"]["dataset"]
    channels = ctx.scratch["context"]["channels"]
    hyperparams = ctx.scratch["context"]["hyperparameters"]
    ensure_sampler_settings(ctx, hyperparams)
    df = pd.DataFrame(dataset)
    X, y, exposures = build_design_matrix(dataset, channels, hyperparams)
    warnings = record_warning(ctx)
    sampler_cfg = hyperparams.get("sampler", {})
    try:
        inference = run_mmm_inference(X, y, channels, hyperparams["priors"], sampler_cfg)
    except Exception as exc:
        message = str(exc)
        lowered = message.lower()
        if PYMC_AVAILABLE and any(token in lowered for token in ("chain", "converg", "diverg")):
            sampler_cfg["chains"] = max(2, int(sampler_cfg.get("chains", 1)) * 2)
            record_warning(ctx, f"Retrying MMM inference with {sampler_cfg['chains']} chains due to convergence warning: {exc}")
            inference = run_mmm_inference(X, y, channels, hyperparams["priors"], sampler_cfg)
        else:
            record_warning(ctx, f"Model inference error: {exc}")
            raise
    business = compute_business_metrics(df, exposures, inference["coefficients"])
    model_outputs = {
        "coefficients": inference["coefficients"],
        "posterior": inference["posterior"],
        "metrics": inference["metrics"],
        "roi": business["roi"],
        "avg_roi": business["avg_roi"],
        "sign_alignment": business["sign_alignment"],
        "saturation_index": business["saturation_index"],
        "spend_totals": business["spend_totals"],
        "predictions": inference["predictions"],
    }
    ctx.scratch["model_outputs"] = model_outputs
    ctx.scratch.setdefault("model_history", []).append(
        {
            "iteration": ctx.scratch.get("iteration", 0),
            "hyperparameters": {k: v for k, v in hyperparams["media"].items()},
            "metrics": model_outputs["metrics"],
            "avg_roi": model_outputs["avg_roi"],
        }
    )
    coeff_summary = ", ".join(f"{channel}={beta:.4f}" for channel, beta in model_outputs["coefficients"].items())
    bus = ctx.get_message_bus()
    bus.publish(
        agent="mmm_modeler",
        content=f"MMM coefficients -> {coeff_summary}",
        iteration=ctx.scratch.get("iteration", 0),
        channel="model",
    )
    return AgentStep(
        out_messages=[Message(role="mmm_modeler", content=f"coefficients: {coeff_summary}")],
        state_updates={"model_outputs": model_outputs},
    )


def evaluation_step(ctx: Context) -> AgentStep:
    model_outputs = ctx.scratch.get("model_outputs")
    if not model_outputs:
        raise RuntimeError("Model outputs unavailable; run the modelling agent first.")
    knowledge = ctx.get_component("knowledge.evaluation")
    metrics = model_outputs["metrics"]
    sign_alignment = model_outputs["sign_alignment"]
    roi = model_outputs["roi"]
    warnings = ctx.scratch.get("warnings", [])
    prompt_lines = [
        "You are the evaluation analyst reviewing MMM diagnostics.",
        f"Metrics: {metrics}",
        f"Sign alignment: {sign_alignment}",
        f"Channel ROI: {roi}",
        f"Operational warnings: {warnings}",
        f"Heuristic reminders: {knowledge['fit_quality']}",
        f"Expected signs: {knowledge['sign_expectations']}",
        f"Key drivers: {knowledge['drivers']}",
        "Summarise the statistical fit, highlight any business prior conflicts, and flag ROI outliers in three paragraphs.",
    ]
    insights = ctx.llm.generate("".join(prompt_lines))
    diagnostics = {"metrics": metrics, "sign_alignment": sign_alignment, "roi": roi, "insights": insights, "warnings": warnings}
    ctx.scratch["evaluation"] = diagnostics
    bus = ctx.get_message_bus()
    bus.publish(agent="mmm_evaluator", content=insights, iteration=ctx.scratch.get("iteration", 0), channel="evaluation")
    return AgentStep(out_messages=[Message(role="mmm_evaluator", content=insights)], state_updates={"evaluation": diagnostics})


def recommendation_step(ctx: Context) -> AgentStep:
    model_outputs = ctx.scratch.get("model_outputs")
    evaluation = ctx.scratch.get("evaluation")
    knowledge = ctx.get_component("knowledge.recommendation")
    if not model_outputs or not evaluation:
        raise RuntimeError("Model and evaluation outputs required before recommendations.")
    warnings = ctx.scratch.get("warnings", [])
    prompt_lines = [
        "You are the marketing strategist turning MMM results into actions.",
        f"ROI: {model_outputs['roi']}",
        f"Saturation index: {model_outputs['saturation_index']}",
        f"Spend totals: {model_outputs['spend_totals']}",
        f"Evaluation narrative: {evaluation['insights']}",
        f"Budgeting heuristics: {knowledge['budgeting']}",
        f"Operational levers: {knowledge['operations']}",
        f"Extra heuristics: {knowledge['heuristics']}",
        f"Known warnings: {warnings}",
        "Return a numbered action list covering budget shifts, channel experiments, and diagnostic follow-ups.",
    ]
    guidance = ctx.llm.generate("".join(prompt_lines))
    ctx.scratch["recommendations"] = guidance
    entry = {
        "iteration": ctx.scratch.get("iteration", 0),
        "hyperparameters": dict(ctx.scratch["context"]["hyperparameters"]["media"]),
        "recommendation": guidance,
    }
    ctx.scratch.setdefault("recommendation_history", []).append(entry)
    bus = ctx.get_message_bus()
    bus.publish(agent="mmm_advisor", content=guidance, iteration=ctx.scratch.get("iteration", 0), channel="recommendation")
    return AgentStep(out_messages=[Message(role="mmm_advisor", content=guidance)], state_updates={"recommendations": guidance})


class MarketingCompositeTrainer:
    """Reinforcement learning controller using a composite score of fit and business heuristics."""

    def setup(self, context: Context, *, environment: Mapping[str, Any] | None = None) -> None:
        context.scratch.setdefault("rl_history", [])
        context.scratch.setdefault("rl_scores", [])

    def update(self, context: Context, step: AgentStep, *, environment: Mapping[str, Any] | None = None) -> None:
        env = environment or {}
        model_outputs = context.scratch.get("model_outputs", {})
        metrics = model_outputs.get("metrics", {})
        avg_roi = float(model_outputs.get("avg_roi", 0.0))
        sign_alignment = model_outputs.get("sign_alignment", {})
        weights = env.get("weights", {"stats": 0.6, "roi": 0.25, "sign": 0.15})
        rmse = float(metrics.get("rmse", env.get("rmse_baseline", 100.0)))
        stats_component = max(0.0, 1.0 - rmse / env.get("rmse_baseline", 100.0))
        roi_component = min(1.0, avg_roi / env.get("roi_target", 1.5))
        sign_penalty = sum(0 for aligned in sign_alignment.values() if aligned) + sum(1 for aligned in sign_alignment.values() if not aligned)
        sign_component = max(0.0, 1.0 - 0.3 * sign_penalty)
        score = (
            weights.get("stats", 0.6) * stats_component
            + weights.get("roi", 0.25) * roi_component
            + weights.get("sign", 0.15) * sign_component
        )
        context.scratch.setdefault("rl_scores", []).append(
            {
                "iteration": context.scratch.get("iteration", 0),
                "score": round(score, 4),
                "components": {
                    "stats": round(stats_component, 4),
                    "roi": round(roi_component, 4),
                    "sign": round(sign_component, 4),
                },
            }
        )
        hyperparams = context.scratch["context"]["hyperparameters"]["media"]
        adstock = hyperparams["adstock_rate"]
        alpha = hyperparams["hill_alpha"]
        gamma = hyperparams["hill_gamma"]
        target = env.get("target_score", 0.8)
        step_sizes = env.get("step_sizes", {"adstock": 0.05, "alpha": 0.1, "gamma": 0.05})
        improved = score >= target
        if not improved:
            if stats_component < 0.8:
                adstock = min(env.get("adstock_bounds", (0.2, 0.9))[1], adstock + step_sizes.get("adstock", 0.05))
            if roi_component < 0.9:
                alpha = max(env.get("hill_alpha_bounds", (0.8, 2.2))[0], alpha - step_sizes.get("alpha", 0.1))
                gamma = max(env.get("hill_gamma_bounds", (0.3, 1.2))[0], gamma - step_sizes.get("gamma", 0.05))
            if sign_component < 0.9:
                alpha = min(env.get("hill_alpha_bounds", (0.8, 2.2))[1], alpha + step_sizes.get("alpha", 0.1))
        else:
            adstock = max(env.get("adstock_bounds", (0.2, 0.9))[0], adstock - step_sizes.get("adstock", 0.03))
            gamma = min(env.get("hill_gamma_bounds", (0.3, 1.2))[1], gamma + step_sizes.get("gamma", 0.03))
        hyperparams.update({"adstock_rate": round(adstock, 3), "hill_alpha": round(alpha, 3), "hill_gamma": round(gamma, 3)})
        record_warning(context, None)
        record = {
            "iteration": context.scratch.get("iteration", 0),
            "score": round(score, 4),
            "adstock_rate": hyperparams["adstock_rate"],
            "hill_alpha": hyperparams["hill_alpha"],
            "hill_gamma": hyperparams["hill_gamma"],
        }
        context.scratch.setdefault("rl_history", []).append(record)
        bus = context.get_message_bus()
        bus.publish(
            agent="mmm_rl_trainer",
            content=f"RL score={score:.3f} -> adstock={hyperparams['adstock_rate']} alpha={hyperparams['hill_alpha']} gamma={hyperparams['hill_gamma']}",
            iteration=context.scratch.get("iteration", 0),
            channel="rl",
        )



def summary_step(ctx: Context) -> AgentStep:
    model_outputs = ctx.scratch.get("model_outputs", {})
    recommendation_history = ctx.scratch.get("recommendation_history", [])
    model_history = ctx.scratch.get("model_history", [])
    rl_scores = ctx.scratch.get("rl_scores", [])
    rl_history = ctx.scratch.get("rl_history", [])
    latest_evaluation = ctx.scratch.get("evaluation", {})
    latest_metrics = model_outputs.get("metrics", {})
    avg_roi = model_outputs.get("avg_roi", 0.0)
    warnings = ctx.scratch.get("warnings", [])
    prompt_lines = [
        "You are the executive sponsor summarising the marketing mix modelling run.",
        f"Latest fit metrics: {latest_metrics}",
        f"Average ROI: {avg_roi}",
        f"Recent composite scores: {rl_scores[-3:] if rl_scores else []}",
        f"Recent RL adjustments: {rl_history[-3:] if rl_history else []}",
        f"Recent model iterations: {model_history[-3:] if model_history else []}",
        f"Recommendation excerpts: {[entry['recommendation'][:160] for entry in recommendation_history[-3:]] if recommendation_history else []}",
        f"Evaluation narrative: {latest_evaluation.get('insights', 'n/a')}",
        f"Current warnings: {warnings}",
        "Produce a clear explanation that states what was done, why the resulting model is strong, and how the recommendations follow from evidence.",
        "Structure the response with sections for overview, statistical evidence, business justification, and recommended next moves.",
    ]
    explanation = ctx.llm.generate("".join(prompt_lines)) if ctx.llm else "Summary unavailable: no LLM adapter."
    summary_payload = {
        "explanation": explanation,
        "evidence": {
            "metrics": latest_metrics,
            "avg_roi": avg_roi,
            "recent_scores": rl_scores[-3:] if rl_scores else [],
            "recent_rl_steps": rl_history[-3:] if rl_history else [],
            "recent_model_iterations": model_history[-3:] if model_history else [],
            "recent_recommendations": recommendation_history[-3:] if recommendation_history else [],
            "warnings": warnings,
        },
    }
    ctx.scratch["final_summary"] = summary_payload
    try:
        bus = ctx.get_message_bus()
        bus.publish(
            agent="mmm_summary_agent",
            content=explanation,
            iteration=ctx.scratch.get("iteration", 0),
            channel="summary",
        )
    except KeyError:
        pass
    return AgentStep(
        out_messages=[Message(role="mmm_summary_agent", content=explanation)],
        state_updates={"final_summary": summary_payload},
    )


In [40]:
# Cell 3: evaluation, recommendations, and reinforcement learning trainer

def evaluation_step(ctx: Context) -> AgentStep:
    model_outputs = ctx.scratch.get("model_outputs")
    if not model_outputs:
        raise RuntimeError("Model outputs unavailable; run the modelling agent first.")
    knowledge = ctx.get_component("knowledge.evaluation")
    metrics = model_outputs["metrics"]
    sign_alignment = model_outputs["sign_alignment"]
    roi = model_outputs["roi"]
    prompt_lines = [
        "You are the evaluation analyst reviewing MMM diagnostics.",
        f"Metrics: {metrics}",
        f"Sign alignment: {sign_alignment}",
        f"Channel ROI: {roi}",
        f"Heuristic reminders: {knowledge['fit_quality']}",
        f"Expected signs: {knowledge['sign_expectations']}",
        f"Key drivers: {knowledge['drivers']}",
        "Summarise the statistical fit, highlight any business prior conflicts, and flag ROI outliers in three paragraphs.",
    ]
    insights = ctx.llm.generate("".join(prompt_lines))
    diagnostics = {"metrics": metrics, "sign_alignment": sign_alignment, "roi": roi, "insights": insights}
    ctx.scratch["evaluation"] = diagnostics
    bus = ctx.get_message_bus()
    bus.publish(agent="mmm_evaluator", content=insights, iteration=ctx.scratch.get("iteration", 0), channel="evaluation")
    return AgentStep(out_messages=[Message(role="mmm_evaluator", content=insights)], state_updates={"evaluation": diagnostics})

def recommendation_step(ctx: Context) -> AgentStep:
    model_outputs = ctx.scratch.get("model_outputs")
    evaluation = ctx.scratch.get("evaluation")
    knowledge = ctx.get_component("knowledge.recommendation")
    if not model_outputs or not evaluation:
        raise RuntimeError("Model and evaluation outputs required before recommendations.")
    prompt_lines = [
        "You are the marketing strategist turning MMM results into actions.",
        f"ROI: {model_outputs['roi']}",
        f"Saturation index: {model_outputs['saturation_index']}",
        f"Spend totals: {model_outputs['spend_totals']}",
        f"Evaluation narrative: {evaluation['insights']}",
        f"Budgeting heuristics: {knowledge['budgeting']}",
        f"Operational levers: {knowledge['operations']}",
        f"Extra heuristics: {knowledge['heuristics']}",
        "Return a numbered action list covering budget shifts, channel experiments, and diagnostic follow-ups.",
    ]
    guidance = ctx.llm.generate("".join(prompt_lines))
    ctx.scratch["recommendations"] = guidance
    entry = {
        "iteration": ctx.scratch.get("iteration", 0),
        "hyperparameters": dict(ctx.scratch["context"]["hyperparameters"]["media"]),
        "recommendation": guidance,
    }
    ctx.scratch.setdefault("recommendation_history", []).append(entry)
    bus = ctx.get_message_bus()
    bus.publish(agent="mmm_advisor", content=guidance, iteration=ctx.scratch.get("iteration", 0), channel="recommendation")
    return AgentStep(out_messages=[Message(role="mmm_advisor", content=guidance)], state_updates={"recommendations": guidance})

class MarketingCompositeTrainer:
    """Reinforcement learning controller using a composite score of fit and business heuristics."""

    def setup(self, context: Context, *, environment: Mapping[str, Any] | None = None) -> None:
        context.scratch.setdefault("rl_history", [])
        context.scratch.setdefault("rl_scores", [])

    def update(self, context: Context, step: AgentStep, *, environment: Mapping[str, Any] | None = None) -> None:
        env = environment or {}
        model_outputs = context.scratch.get("model_outputs", {})
        metrics = model_outputs.get("metrics", {})
        avg_roi = float(model_outputs.get("avg_roi", 0.0))
        sign_alignment = model_outputs.get("sign_alignment", {})
        weights = env.get("weights", {"stats": 0.6, "roi": 0.25, "sign": 0.15})
        rmse = float(metrics.get("rmse", env.get("rmse_baseline", 100.0)))
        stats_component = max(0.0, 1.0 - rmse / env.get("rmse_baseline", 100.0))
        roi_component = min(1.0, avg_roi / env.get("roi_target", 1.5))
        sign_penalty = sum(0 for aligned in sign_alignment.values() if aligned) + sum(1 for aligned in sign_alignment.values() if not aligned)
        sign_component = max(0.0, 1.0 - 0.3 * sign_penalty)
        score = (
            weights.get("stats", 0.6) * stats_component
            + weights.get("roi", 0.25) * roi_component
            + weights.get("sign", 0.15) * sign_component
        )
        context.scratch.setdefault("rl_scores", []).append({
            "iteration": context.scratch.get("iteration", 0),
            "score": round(score, 4),
            "components": {
                "stats": round(stats_component, 4),
                "roi": round(roi_component, 4),
                "sign": round(sign_component, 4),
            },
        })
        hyperparams = context.scratch["context"]["hyperparameters"]["media"]
        adstock = hyperparams["adstock_rate"]
        alpha = hyperparams["hill_alpha"]
        gamma = hyperparams["hill_gamma"]
        target = env.get("target_score", 0.8)
        step_sizes = env.get("step_sizes", {"adstock": 0.05, "alpha": 0.1, "gamma": 0.05})
        improved = score >= target
        if not improved:
            if stats_component < 0.8:
                adstock = min(env.get("adstock_bounds", (0.2, 0.9))[1], adstock + step_sizes.get("adstock", 0.05))
            if roi_component < 0.9:
                alpha = max(env.get("hill_alpha_bounds", (0.8, 2.2))[0], alpha - step_sizes.get("alpha", 0.1))
                gamma = max(env.get("hill_gamma_bounds", (0.3, 1.2))[0], gamma - step_sizes.get("gamma", 0.05))
            if sign_component < 0.9:
                alpha = min(env.get("hill_alpha_bounds", (0.8, 2.2))[1], alpha + step_sizes.get("alpha", 0.1))
        else:
            adstock = max(env.get("adstock_bounds", (0.2, 0.9))[0], adstock - step_sizes.get("adstock", 0.03))
            gamma = min(env.get("hill_gamma_bounds", (0.3, 1.2))[1], gamma + step_sizes.get("gamma", 0.03))
        hyperparams.update({"adstock_rate": round(adstock, 3), "hill_alpha": round(alpha, 3), "hill_gamma": round(gamma, 3)})
        record = {
            "iteration": context.scratch.get("iteration", 0),
            "score": round(score, 4),
            "adstock_rate": hyperparams["adstock_rate"],
            "hill_alpha": hyperparams["hill_alpha"],
            "hill_gamma": hyperparams["hill_gamma"],
        }
        context.scratch.setdefault("rl_history", []).append(record)
        bus = context.get_message_bus()
        bus.publish(
            agent="mmm_rl_trainer",
            content=f"RL score={score:.3f} -> adstock={hyperparams['adstock_rate']} alpha={hyperparams['hill_alpha']} gamma={hyperparams['hill_gamma']}",
            iteration=context.scratch.get("iteration", 0),
            channel="rl",
        )



def summary_step(ctx: Context) -> AgentStep:
    model_outputs = ctx.scratch.get("model_outputs", {})
    recommendation_history = ctx.scratch.get("recommendation_history", [])
    model_history = ctx.scratch.get("model_history", [])
    rl_scores = ctx.scratch.get("rl_scores", [])
    rl_history = ctx.scratch.get("rl_history", [])
    latest_evaluation = ctx.scratch.get("evaluation", {})
    latest_metrics = model_outputs.get("metrics", {})
    avg_roi = model_outputs.get("avg_roi", 0.0)
    prompt_lines = [
        "You are the executive sponsor summarising the marketing mix modelling run.",
        f"Latest fit metrics: {latest_metrics}",
        f"Average ROI: {avg_roi}",
        f"Recent composite scores: {rl_scores[-3:] if rl_scores else []}",
        f"Recent RL adjustments: {rl_history[-3:] if rl_history else []}",
        f"Recent model iterations: {model_history[-3:] if model_history else []}",
        f"Recommendation excerpts: {[entry['recommendation'][:160] for entry in recommendation_history[-3:]] if recommendation_history else []}",
        f"Evaluation narrative: {latest_evaluation.get('insights', 'n/a')}",
        "Produce a clear explanation that states what was done, why the resulting model is strong, and how the recommendations follow from evidence.",
        "Structure the response with sections for overview, statistical evidence, business justification, and recommended next moves.",
    ]
    explanation = ctx.llm.generate("".join(prompt_lines)) if ctx.llm else "Summary unavailable: no LLM adapter."
    summary_payload = {
        "explanation": explanation,
        "evidence": {
            "metrics": latest_metrics,
            "avg_roi": avg_roi,
            "recent_scores": rl_scores[-3:] if rl_scores else [],
            "recent_rl_steps": rl_history[-3:] if rl_history else [],
            "recent_model_iterations": model_history[-3:] if model_history else [],
            "recent_recommendations": recommendation_history[-3:] if recommendation_history else [],
        },
    }
    ctx.scratch["final_summary"] = summary_payload
    try:
        bus = ctx.get_message_bus()
        bus.publish(
            agent="mmm_summary_agent",
            content=explanation,
            iteration=ctx.scratch.get("iteration", 0),
            channel="summary",
        )
    except KeyError:
        pass
    return AgentStep(
        out_messages=[Message(role="mmm_summary_agent", content=explanation)],
        state_updates={"final_summary": summary_payload},
    )


In [41]:
# Cell 4: assemble agents and run iterative pipeline

def run_pipeline(iterations: int = 4, rl_environment: Mapping[str, Any] | None = None) -> Mapping[str, Any]:
    bus, bus_capability = build_message_bus()
    context_capability = build_context_capability()
    evaluation_resource = build_evaluation_resource()
    recommendation_resource = build_recommendation_resource()

    rl_trainer = MarketingCompositeTrainer()

    modeling_agent = (
        AgentBuilder(name="mmm_modeler", role="modeler")
        .with_capabilities([bus_capability, context_capability])
        .with_reinforcement_learning(rl_trainer, environment=rl_environment or MARKETING_RL_ENVIRONMENT)
        .with_step(modelling_step)
        .build()
    )

    evaluation_agent = (
        AgentBuilder(name="mmm_evaluator", role="analyst")
        .with_capabilities([bus_capability, context_capability, evaluation_resource])
        .with_llm(llm_adapter)
        .with_step(evaluation_step)
        .build()
    )

    recommendation_agent = (
        AgentBuilder(name="mmm_advisor", role="strategist")
        .with_capabilities([bus_capability, context_capability, recommendation_resource])
        .with_llm(llm_adapter)
        .with_step(recommendation_step)
        .build()
    )

    summary_agent = (
        AgentBuilder(name="mmm_summary_agent", role="sponsor")
        .with_capabilities([bus_capability])
        .with_llm(llm_adapter)
        .with_step(summary_step)
        .build()
    )

    context = Context(goal="Optimise marketing mix with composite RL score")
    context.add_component("message_bus", bus)

    for iteration in range(iterations):
        context.scratch["iteration"] = iteration
        for agent in (modeling_agent, evaluation_agent, recommendation_agent):
            result = agent.run(context)
            if result.out_messages:
                context.push_message(result.out_messages[-1])

    summary_result = summary_agent.run(context)
    if summary_result.out_messages:
        context.push_message(summary_result.out_messages[-1])

    model_history = context.scratch.get("model_history", [])
    rl_history = context.scratch.get("rl_history", [])
    scores = context.scratch.get("rl_scores", [])
    best = max(scores, key=lambda item: item["score"], default=None)
    final_summary = context.scratch.get("final_summary", {})

    return {
        "model_history": model_history,
        "rl_history": rl_history,
        "scores": scores,
        "best_iteration": best,
        "final_metrics": context.scratch.get("model_outputs", {}).get("metrics", {}),
        "final_roi": context.scratch.get("model_outputs", {}).get("roi", {}),
        "final_summary": final_summary,
        "recommendation_history": context.scratch.get("recommendation_history", []),
        "message_bus": [
            {"agent": record.agent, "channel": record.channel, "message": record.content[:160]}
            for record in bus.conversation()
        ],
        "latest_recommendation": context.scratch.get("recommendations"),
        "summary_message": summary_result.out_messages[-1].content if summary_result.out_messages else "",
    }

PIPELINE_REPORT = run_pipeline(iterations=4, rl_environment=MARKETING_RL_ENVIRONMENT)
PIPELINE_REPORT


Multiprocess sampling (2 chains in 2 jobs)
CompoundStep
>Metropolis: [intercept]
>Metropolis: [sigma]
>Metropolis: [search]
>Metropolis: [social]
>Metropolis: [tv]
>Metropolis: [affiliate]
/opt/anaconda3/lib/python3.12/site-packages/pymc/step_methods/metropolis.py:321: RuntimeWarning: overflow encountered in exp
  "accept": np.mean(np.exp(self.accept_rate_iter)),
/opt/anaconda3/lib/python3.12/site-packages/pymc/step_methods/metropolis.py:321: RuntimeWarning: overflow encountered in exp
  "accept": np.mean(np.exp(self.accept_rate_iter)),
Sampling 2 chains for 500 tune and 1_000 draw iterations (1_000 + 2_000 draws total) took 1 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable 

{'model_history': [{'iteration': 0,
   'hyperparameters': {'adstock_rate': 0.45,
    'hill_alpha': 1.25,
    'hill_gamma': 0.7},
   'metrics': {'rmse': 78.57335652494798,
    'mae': 66.4495696898586,
    'mape': 8.10831464213127,
    'r2': -0.0016978374603375812},
   'avg_roi': 0.7728039974257879},
  {'iteration': 1,
   'hyperparameters': {'adstock_rate': 0.49,
    'hill_alpha': 1.15,
    'hill_gamma': 0.65},
   'metrics': {'rmse': 78.55895643283945,
    'mae': 66.45669639296452,
    'mape': 8.112820957776169,
    'r2': -0.0013307099765764718},
   'avg_roi': 0.7573340117344536},
  {'iteration': 2,
   'hyperparameters': {'adstock_rate': 0.53,
    'hill_alpha': 1.05,
    'hill_gamma': 0.6},
   'metrics': {'rmse': 78.54468547140571,
    'mae': 66.46480309486816,
    'mape': 8.117948721315617,
    'r2': -0.0009669410406765166},
   'avg_roi': 0.7978901414125795},
  {'iteration': 3,
   'hyperparameters': {'adstock_rate': 0.57,
    'hill_alpha': 0.95,
    'hill_gamma': 0.55},
   'metrics': {'

In [42]:
# Cell 5: inspect composite score trajectory and recommendations

{
    "rl_scores": PIPELINE_REPORT["scores"],
    "best_iteration": PIPELINE_REPORT["best_iteration"],
    "final_metrics": PIPELINE_REPORT["final_metrics"],
    "final_roi": PIPELINE_REPORT["final_roi"],
    "latest_recommendation_excerpt": PIPELINE_REPORT["latest_recommendation"][:240] if PIPELINE_REPORT["latest_recommendation"] else None,
    "summary_excerpt": PIPELINE_REPORT.get("final_summary", {}).get("explanation", "")[:280],
}


{'rl_scores': [{'iteration': 0,
   'score': 0.2815,
   'components': {'stats': 0.0178, 'roi': 0.483, 'sign': 1.0}},
  {'iteration': 1,
   'score': 0.2791,
   'components': {'stats': 0.018, 'roi': 0.4733, 'sign': 1.0}},
  {'iteration': 2,
   'score': 0.2856,
   'components': {'stats': 0.0182, 'roi': 0.4987, 'sign': 1.0}},
  {'iteration': 3,
   'score': 0.2855,
   'components': {'stats': 0.018, 'roi': 0.4989, 'sign': 1.0}}],
 'best_iteration': {'iteration': 2,
  'score': 0.2856,
  'components': {'stats': 0.0182, 'roi': 0.4987, 'sign': 1.0}},
 'final_metrics': {'rmse': 78.55850391311235,
  'mae': 66.4568979825922,
  'mape': 8.11295610565134,
  'r2': -0.0013191741666536139},
 'final_roi': {'search': 0.9244957119524775,
  'social': 0.9989863590396406,
  'tv': 0.5537713920964796,
  'affiliate': 0.7154319497425323},
 'latest_recommendation_excerpt': 'Based on the provided marketing mix model (MMM) results and the evaluation narrative, here is a structured action list to optimize marketing str

In [43]:
# Cell 6: detailed recommendation history for audit

[
    {
        "iteration": entry["iteration"],
        "hyperparameters": entry["hyperparameters"],
        "excerpt": entry["recommendation"][:200],
    }
    for entry in PIPELINE_REPORT["recommendation_history"]
]


[{'iteration': 0,
  'hyperparameters': {'adstock_rate': 0.49,
   'hill_alpha': 1.15,
   'hill_gamma': 0.65},
  'excerpt': 'Based on the provided marketing mix model (MMM) results and the evaluation narrative, here is a structured action list to optimize the marketing strategy:\n\n### Action List\n\n1. **Budget Reallocation:**'},
 {'iteration': 1,
  'hyperparameters': {'adstock_rate': 0.53,
   'hill_alpha': 1.05,
   'hill_gamma': 0.6},
  'excerpt': 'Based on the provided marketing mix model (MMM) results and the evaluation narrative, here is a structured action list to optimize marketing efforts:\n\n### Action List\n\n1. **Budget Reallocation:**\n   -'},
 {'iteration': 2,
  'hyperparameters': {'adstock_rate': 0.57,
   'hill_alpha': 0.95,
   'hill_gamma': 0.55},
  'excerpt': 'Based on the provided MMM results, evaluation narrative, and heuristics, here is a structured action list to optimize the marketing strategy:\n\n### Action List\n\n1. **Budget Reallocation:**\n   - **Shift'},
 {'ite

In [44]:
# Cell 7: final summary with evidence

{
    "explanation": PIPELINE_REPORT.get("final_summary", {}).get("explanation", ""),
    "evidence": PIPELINE_REPORT.get("final_summary", {}).get("evidence", {}),
}


{'explanation': "# Marketing Mix Modelling Summary\n\n## Overview\nThe recent marketing mix modelling (MMM) run aimed to evaluate the effectiveness of our marketing channels and optimize our marketing strategy. The model produced several key metrics that provide insights into the performance of our marketing investments. While the model has identified some positive contributions from various channels, it also highlights significant areas for improvement, particularly in TV and affiliate marketing.\n\n## Statistical Evidence\nThe latest fit metrics from the model are as follows:\n- **Root Mean Square Error (RMSE):** 78.56\n- **Mean Absolute Error (MAE):** 66.46\n- **Mean Absolute Percentage Error (MAPE):** 8.11%\n- **R-squared (R²):** -0.0013\n\nThese metrics indicate that the model's predictions deviate significantly from actual outcomes, suggesting a poor fit. The MAPE is marginally acceptable but close to the upper threshold of 10%, while the negative R² value indicates that the mode

In [45]:
# Cell 8: top marketing mix models by composite score

from typing import List, Dict, Mapping, Any

def summarize_top_models(report: Mapping[str, Any], top_n: int = 3) -> List[Dict[str, Any]]:
    scores = sorted(report.get("scores", []), key=lambda item: item["score"], reverse=True)
    history_map = {entry["iteration"]: entry for entry in report.get("model_history", [])}
    recommendation_map = {entry["iteration"]: entry for entry in report.get("recommendation_history", [])}
    top_models: List[Dict[str, Any]] = []
    for item in scores[:top_n]:
        iteration = item["iteration"]
        metrics_record = history_map.get(iteration, {})
        metrics = metrics_record.get("metrics", {})
        roi_value = metrics_record.get("avg_roi", 0.0)
        recommendation_excerpt = recommendation_map.get(iteration, {}).get("recommendation", "")[:280]
        explanation = (
            f"Iteration {iteration} reached a composite score of {item['score']:.3f} with "
            f"RMSE {metrics.get('rmse', float('nan')):.2f}, ROI {roi_value:.2f}. "
            f"Recommendation focus: {recommendation_excerpt}"
        )
        top_models.append(
            {
                "iteration": iteration,
                "composite_score": round(item["score"], 4),
                "hyperparameters": metrics_record.get("hyperparameters", {}),
                "metrics": metrics,
                "avg_roi": roi_value,
                "recommendation_excerpt": recommendation_excerpt,
                "explanation": explanation,
            }
        )
    return top_models

TOP_MMM_MODELS = summarize_top_models(PIPELINE_REPORT, top_n=3)
TOP_MMM_MODELS


[{'iteration': 2,
  'composite_score': 0.2856,
  'hyperparameters': {'adstock_rate': 0.53,
   'hill_alpha': 1.05,
   'hill_gamma': 0.6},
  'metrics': {'rmse': 78.54468547140571,
   'mae': 66.46480309486816,
   'mape': 8.117948721315617,
   'r2': -0.0009669410406765166},
  'avg_roi': 0.7978901414125795,
  'recommendation_excerpt': 'Based on the provided MMM results, evaluation narrative, and heuristics, here is a structured action list to optimize the marketing strategy:\n\n### Action List\n\n1. **Budget Reallocation:**\n   - **Shift Budget from TV and Affiliate:** Reduce spending on TV (currently at $13,690) an',
  'explanation': 'Iteration 2 reached a composite score of 0.286 with RMSE 78.54, ROI 0.80. Recommendation focus: Based on the provided MMM results, evaluation narrative, and heuristics, here is a structured action list to optimize the marketing strategy:\n\n### Action List\n\n1. **Budget Reallocation:**\n   - **Shift Budget from TV and Affiliate:** Reduce spending on TV (curr